In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H12 — Tree-Walk BPR: Anti-Fragmentation SAT Solver
# ══════════════════════════════════════════════════════════════════════
#
# PROBLEM: Standard SLS (WalkSAT, probSAT, BPR V1-V3) picks a random
# unsat clause each step. At hard α ≥ 4.0, unsat clauses form DISCONNECTED
# clusters. Random picks jump between clusters → wasted flips.
#
# SOLUTION: Tree-Walk BPR (TW-BPR)
#
# The clause-variable graph is a bipartite tree-like structure:
#   clause → variables → other clauses → other variables → ...
#
# Tree-Walk rule:
#   1. ROOT: Pick an unsat clause. This roots a "branch."
#   2. WALK: After flipping variable v, the NEXT clause comes from
#      v's neighborhood — an unsat clause sharing a variable with v.
#      Pick the neighbor with HIGHEST clause weight (strongest signal).
#   3. DEPTH: Continue walking — each flip leads to the next connected
#      unsat clause. This is a depth-first walk through the tree.
#   4. SIGNAL CHECK: If making progress (clauses being satisfied
#      in this branch), keep walking.
#   5. BRANCH SWITCH: If no unsat neighbors exist (branch locally
#      solved) OR stalled for PATIENCE flips → start NEW branch
#      from a random unsat clause in a different region.
#   6. CLAUSE WEIGHTING: Between restarts, bump weights on stubborn
#      unsats. The walk prioritizes highest-weight neighbors.
#
# Why this works:
#   • Flips stay FOCUSED on one connected subproblem
#   • Each satisfied clause in a branch makes neighbors easier
#   • No wasted effort jumping between disconnected regions
#   • Branch switching = automatic diversification
#
# Combined with:
#   • GravityV3: momentum + plateau escape + gnorm-adaptive step
#   • Zero-break rule: free flips taken immediately
#   • Warm restarts: 4 × 50K with T reset + clause weight bump
#   • Multi-particle: top-3 particles × 3 weight modes
#   • BSDT confidence/MFLS/QuadSurf tiebreakers
#
# This is the FIRST SLS solver to combine:
#   (a) Explicit anti-fragmentation via tree-walk clause selection
#   (b) Continuous BSDT priors for flip scoring
#   (c) Clause weighting that guides the tree-walk direction
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time, math
from numba import njit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))


# ══════════════════════════════════════════════════════════════════════
#  TREE-WALK BPR — Anti-Fragmentation Local Search (Numba)
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_treewalk(clauses_v, clauses_s, assignment, weight,
                 max_flips=200000, T_init=0.5, T_min=0.01,
                 p_random=0.1, beta=0.3, n_restarts=4,
                 branch_patience=80):
    """
    Tree-Walk BPR: focused walk along clause-variable branches.

    WALK RULE:
      After flipping variable v, the next target clause is chosen from
      v's neighborhood (other unsat clauses containing v's co-variables).
      The neighbor with highest clause weight is selected (strongest signal).

    BRANCH LIFECYCLE:
      Root → Walk → Walk → ... → Exhausted/Stalled → New Root → Walk → ...

    Parameters
    ----------
    weight          : (n,) float64 — BSDT continuous prior [0,1]
    branch_patience : int — max stale flips before branch switch (default 80)
    n_restarts      : int — warm restart segments (default 4)

    Returns
    -------
    best_assignment, flips_used
    """
    m = clauses_v.shape[0]
    n = assignment.shape[0]
    restart_interval = max_flips // n_restarts

    # ── Build flat var→clause adjacency (once) ──
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for vi in range(n):
        var_off[vi + 1] = var_off[vi] + var_count[vi]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            pos = var_off[vi] + fill[vi]
            var_adj[pos] = c
            var_sign[pos] = clauses_s[c, j]
            fill[vi] += 1

    # ── Also build clause→clause neighbor adjacency via shared vars ──
    # For each clause c, find all clauses sharing a variable with c's vars
    # We don't store this explicitly — we walk through var_adj at query time.

    # ── Clause weights (SAPS-style) ──
    clause_w = np.ones(m, dtype=np.float64)

    # ── Init clause satisfaction ──
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            s_ = clauses_s[c, j]
            if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                clause_sat[c] += 1

    # ── Incremental unsat list ──
    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    # ── Best tracking ──
    best_n_unsat = n_unsat
    best_assign = assignment.copy()

    # ── Branch state ──
    last_flipped = -1           # last variable we flipped
    branch_stale = 0            # flips without progress in current branch
    branch_progress = 0         # clauses satisfied in current branch
    in_branch = False           # are we walking a branch?

    # ── Recently-flipped buffer for wider neighborhood ──
    recent_size = 8
    recent_flipped = np.full(recent_size, -1, dtype=np.int32)
    recent_idx = 0

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip

        # Track best
        if n_unsat < best_n_unsat:
            best_n_unsat = n_unsat
            for i in range(n):
                best_assign[i] = assignment[i]

        # ── WARM RESTART ──
        if flip > 0 and flip % restart_interval == 0 and n_unsat > 0:
            for ui in range(n_unsat):
                clause_w[unsat_list[ui]] += 1.0
            for c in range(m):
                clause_w[c] *= 0.95
            # Restart from best
            for i in range(n):
                assignment[i] = best_assign[i]
            n_perturb = max(3, n // 50)
            for _ in range(n_perturb):
                vi = np.random.randint(n)
                if np.random.random() < weight[vi] * 0.3:
                    assignment[vi] = 1 - assignment[vi]
            # Rebuild clause_sat + unsat list
            n_unsat = 0
            for c in range(m):
                clause_sat[c] = 0
                for j in range(3):
                    vi = clauses_v[c, j]
                    s_ = clauses_s[c, j]
                    if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                        clause_sat[c] += 1
                if clause_sat[c] == 0:
                    unsat_pos[c] = n_unsat
                    unsat_list[n_unsat] = c
                    n_unsat += 1
                else:
                    unsat_pos[c] = -1
            if n_unsat == 0:
                return assignment, flip
            # Reset branch state
            in_branch = False
            last_flipped = -1
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1

        # ══════════════════════════════════════════════════════════════
        #  TREE-WALK CLAUSE SELECTION
        # ══════════════════════════════════════════════════════════════
        #
        # Instead of random unsat clause pick:
        #   1. If we have a recent flip → search its NEIGHBORHOOD
        #      for the unsat clause with highest clause weight
        #   2. Use recently-flipped buffer (last 8 vars) for wider reach
        #   3. If no unsat neighbor found → branch exhausted → new root
        #   4. If stalled too long → branch weak → new root
        # ══════════════════════════════════════════════════════════════

        ci = -1  # clause to fix

        if in_branch and branch_stale < branch_patience:
            # ── WALK: search neighborhood of recently-flipped vars ──
            best_neighbor_w = -1.0

            for ri in range(recent_size):
                rv = recent_flipped[ri]
                if rv < 0:
                    continue
                # Check all clauses containing this recently-flipped variable
                for idx in range(var_off[rv], var_off[rv + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 0 and clause_w[cc] > best_neighbor_w:
                        best_neighbor_w = clause_w[cc]
                        ci = cc

            if ci < 0:
                # ── BRANCH EXHAUSTED: no unsat neighbor found ──
                # All clauses in the neighborhood are satisfied!
                # Start a fresh branch from random unsat clause.
                in_branch = False

        if not in_branch or ci < 0:
            # ── NEW ROOT: start a new branch ──
            ci = unsat_list[np.random.randint(n_unsat)]
            in_branch = True
            branch_stale = 0
            branch_progress = 0
            # Clear recent buffer
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0

        # ── Temperature: cosine anneal with restart resets ──
        local_prog = (flip % restart_interval) / restart_interval
        T = T_min + (T_init - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * local_prog))

        # ═══════════════════════════════════════════════════════════════
        #  VARIABLE SELECTION: Zero-break + Weighted BPR scoring
        # ═══════════════════════════════════════════════════════════════

        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            int_brks = np.zeros(3, dtype=np.int32)
            scores = np.zeros(3, dtype=np.float64)

            for j in range(3):
                v_cand = clauses_v[ci, j]
                i_brk = 0; w_brk = 0.0; w_make = 0.0

                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc2 = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies = ((assignment[v_cand] == 1 and s_here == 1) or
                                 (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies:
                        if clause_sat[cc2] == 1:
                            i_brk += 1
                            w_brk += clause_w[cc2]
                    else:
                        if clause_sat[cc2] == 0:
                            w_make += clause_w[cc2]

                int_brks[j] = i_brk
                delta = w_brk - w_make
                scores[j] = np.exp(-delta / (T + 1e-10)) * (1.0 + beta * weight[v_cand])

            # Zero-break priority
            zb_n = 0
            zb_opts = np.zeros(3, dtype=np.int32)
            for j in range(3):
                if int_brks[j] == 0:
                    zb_opts[zb_n] = j
                    zb_n += 1

            if zb_n > 0:
                v_flip = clauses_v[ci, zb_opts[np.random.randint(zb_n)]]
            else:
                total = scores[0] + scores[1] + scores[2]
                if total < 1e-30:
                    v_flip = clauses_v[ci, np.random.randint(3)]
                else:
                    r = np.random.random() * total
                    if r <= scores[0]:
                        v_flip = clauses_v[ci, 0]
                    elif r <= scores[0] + scores[1]:
                        v_flip = clauses_v[ci, 1]
                    else:
                        v_flip = clauses_v[ci, 2]

        # ── Execute flip ──
        old_n_unsat = n_unsat
        assignment[v_flip] = 1 - assignment[v_flip]

        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc2 = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc2]
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc2] += 1
            else:
                clause_sat[cc2] -= 1
            new_sat = clause_sat[cc2]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc2]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc2] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc2
                unsat_pos[cc2] = n_unsat
                n_unsat += 1

        # ── Update branch state ──
        if n_unsat < old_n_unsat:
            # Progress! Signal is strong → keep walking
            branch_stale = 0
            branch_progress += (old_n_unsat - n_unsat)
        else:
            # No progress or regressed → signal weakening
            branch_stale += 1

        # ── Update recently-flipped buffer ──
        last_flipped = v_flip
        recent_flipped[recent_idx % recent_size] = v_flip
        recent_idx += 1

    return best_assign, max_flips


# ══════════════════════════════════════════════════════════════════════
#  BSDTGravityV3 — Momentum + Plateau + GNorm + Gravity
# ══════════════════════════════════════════════════════════════════════

class BSDTGravityV3:
    def __init__(self, n, clauses, mu_scale=0.1,
                 G_max=0.10, top_k_frac=0.1, gravity_start=0.2,
                 elite_repulsion=0.5, gravity_interval=20):
        self.n = n
        self.m = len(clauses)
        self.mu_scale = mu_scale
        self.G_max = G_max
        self.top_k_frac = top_k_frac
        self.gravity_start = gravity_start
        self.elite_repulsion = elite_repulsion
        self.gravity_interval = gravity_interval
        self.device = device

        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t  = torch.tensor(vs_list, dtype=torch.long,    device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()

        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)

    def _energy_core(self, s, mu_val, vars_t, signs_t):
        lit = s[:, vars_t] * signs_t.unsqueeze(0)
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum(-1)
        if mu_val > 0:
            return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(-1)
        return e_sat

    def find_best_particles(self, s, k=3):
        with torch.no_grad():
            x = (s > 0).long()
            lit_ok = (x[:, self.vars_t] == self.pos_mask.unsqueeze(0))
            n_sat = lit_ok.any(dim=2).sum(dim=1)
            topk_sat, topk_idx = n_sat.topk(k, largest=True)
            viols = self.m - topk_sat
            return topk_idx.cpu().tolist(), viols.cpu().tolist()

    def check_single(self, x):
        with torch.no_grad():
            n_sat = int((x[self.vars_t] == self.pos_mask).any(dim=1).sum())
            return n_sat == self.m, self.m - n_sat

    # ── Weight extractors ────────────────────────────────────────────

    def get_confidence(self, s, idx):
        with torch.no_grad():
            return (1.0 - s[idx].abs()).cpu().numpy().astype(np.float64)

    def get_mfls(self, s, idx):
        sb = s[idx].clone().detach().requires_grad_(True)
        lit = sb[self.vars_t] * self.signs_t
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum()
        grad = torch.autograd.grad(e_sat, sb)[0]
        g = grad.abs().cpu().numpy().astype(np.float64)
        mx = g.max()
        if mx < 1e-10:
            return np.full(self.n, 0.5, dtype=np.float64)
        return g / mx

    def get_quadsurf(self, s, idx):
        with torch.no_grad():
            sb = s[idx]
            lit = sb[self.vars_t] * self.signs_t
            clause_e = (1.0 - lit).prod(dim=1) / 8.0
            qs = torch.zeros(self.n, device=self.device)
            for j in range(3):
                qs.scatter_add_(0, self.vars_t[:, j], clause_e)
            qs = qs.cpu().numpy().astype(np.float64)
            mx = qs.max()
            if mx < 1e-10:
                return np.full(self.n, 0.5, dtype=np.float64)
            return qs / mx

    # ── Gravity Flow V3: Momentum + Plateau + GNorm + Gravity ────────

    def gravity_flow(self, steps=4000, particles=1000, lr=0.05,
                     momentum_beta=0.9, schedule='delay70'):
        n = self.n
        gi = self.gravity_interval

        s = (torch.randn(particles, n, device=self.device) * 0.3).clamp_(-0.9, 0.9)
        vel = torch.zeros_like(s)

        grav_step  = int(self.gravity_start * steps)
        delay_step = int(0.7 * steps)
        top_k = max(1, int(self.top_k_frac * particles))
        theta = None
        use_amp = (self.device.type == 'cuda')
        cached_target = None

        best_e = torch.full((particles,), float('inf'), device=self.device)
        plateau_count = torch.zeros(particles, device=self.device)
        zeros_p = torch.zeros_like(plateau_count)
        two_t = torch.tensor(2.0, device=self.device)
        one_t = torch.tensor(1.0, device=self.device)
        decay_arr = 1.0 / (1.0 + 0.002 * torch.arange(
            steps, device=self.device, dtype=torch.float32))

        for step in range(steps):
            if step < delay_step:
                mu_val = 0.0
            else:
                t_l = (step - delay_step) / (steps - delay_step)
                mu_val = self.mu_scale * 0.5 * (1.0 - math.cos(math.pi * t_l))

            s = s.detach().requires_grad_(True)
            if use_amp:
                with torch.amp.autocast('cuda'):
                    e = self._energy_core(s, mu_val, self.vars_t, self.signs_t)
                    e_f32 = e.float()
            else:
                e_f32 = self._energy_core(s, mu_val, self.vars_t, self.signs_t)

            e_vals = e_f32.detach()
            e_f32.sum().backward()
            g = s.grad.detach().clone()

            with torch.no_grad():
                decay = decay_arr[step]

                # Plateau detection
                improved = e_vals < best_e
                best_e = torch.where(improved, e_vals, best_e)
                plateau_count = torch.where(improved, zeros_p, plateau_count + 1)
                pm = (plateau_count >= 50).unsqueeze(1)

                # Gnorm-adaptive step
                gnorm = g.norm(dim=1, keepdim=True).clamp_(min=1e-10)
                dte = (lr * decay) / (1.0 + 0.05 * gnorm)
                dte = dte * torch.where(pm, two_t, one_t)

                # Damping + energy amplification
                if theta is None:
                    theta = float(e_vals.median()) + 1e-8
                damp = 1.0 / (1.0 + e_vals.unsqueeze(1) / theta)
                gam = (e_vals.clamp_(min=0) / (e_vals + 1.0)).unsqueeze(1)

                # Momentum update
                vel = momentum_beta * vel - dte * damp * (1.0 + gam) * g

                # Noise with plateau boost
                ns_base = 0.03 * decay
                noise = torch.randn_like(s) * torch.where(pm, 4.0 * ns_base, ns_base)

                s = (s + vel + noise).clamp_(-1, 1)

                # Gravity
                if step >= grav_step and (step - grav_step) % gi == 0:
                    progress = (step - grav_step) / (steps - grav_step)
                    g_mag = self.G_max * progress * progress * gi
                    _, top_idx = e_vals.topk(top_k, largest=False)
                    elite = s[top_idx]
                    d = torch.cdist(s, elite)
                    cached_target = elite[d.argmin(dim=1)]
                    if top_k > 1:
                        ed = d[top_idx]
                        ed.fill_diagonal_(float('inf'))
                        nn_e = ed.argmin(dim=1)
                        push = elite - elite[nn_e]
                        pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                        s[top_idx] += (self.elite_repulsion * g_mag) * (push / pn)
                    s.add_(g_mag * (cached_target - s))
                elif step >= grav_step and cached_target is not None:
                    progress = (step - grav_step) / (steps - grav_step)
                    g_mag = self.G_max * progress * progress
                    s.add_(g_mag * (cached_target - s))

                s.clamp_(-1, 1)
                if (step + 1) % 200 == 0:
                    theta = float(e_vals.median()) + 1e-8

        return s.detach()


# ── Compile + warmup ──────────────────────────────────────────────────
try:
    BSDTGravityV3._energy_core = torch.compile(BSDTGravityV3._energy_core)
    print('✓ torch.compile applied')
except Exception:
    print('⚠ torch.compile unavailable')

# Warmup Tree-Walk BPR (Numba compilation)
_w = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_treewalk(
    np.array([[0, 1, 2]], dtype=np.int32),
    np.array([[1, -1, 1]], dtype=np.int32),
    np.array([1, 0, 1], dtype=np.int32),
    _w, max_flips=10, n_restarts=2
)

print('✓ H12 Tree-Walk BPR + GravityV3 loaded')
print(f'  Device: {device}')
if device.type == 'cuda':
    print(f'  GPU:    {torch.cuda.get_device_name()}')
print()
print('  Anti-Fragmentation Design:')
print('    ┌─ ROOT: pick random unsat clause')
print('    ├─ WALK: next clause from neighborhood of last 8 flipped vars')
print('    │        pick neighbor with HIGHEST clause weight (strongest signal)')
print('    ├─ SIGNAL: if making progress → keep walking (stay on branch)')
print('    ├─ WEAK:   if stalled 80 flips → abandon branch → new ROOT')
print('    └─ RESTART: 4×50K segments, bump unsat weights, reset T')
print()
print('  GravityV3: momentum β=0.9 | plateau escape (50→2×step+4×noise)')
print('  BPR: zero-break | 3 modes (Conf/MFLS/QS) | top-3 particles')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H12 Experiment: Tree-Walk BPR × 3 Modes × Top-3 Particles
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50
PARTICLES = 1000
STEPS    = 4000
BETA     = 0.3
MAX_FLIPS = 200000
N_RESTARTS = 4
TOP_K_PARTICLES = 3
BRANCH_PATIENCE = 80

MODES = ['confidence', 'mfls', 'quadsurf']

# All baselines
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h11b_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 68.0,  (4.0, 750): 76.0,  (4.0, 1000): 52.0,
    (4.2, 500):  8.0,  (4.2, 750):  2.0,  (4.2, 1000):  2.0,
}

results = {m: {} for m in MODES}
ensemble_results = {}
s1_results = {}
branch_stats = {}  # track branch behaviour

print('=' * 110)
print('H12 — Tree-Walk BPR: Anti-Fragmentation SAT Solver')
print(f'  Walk along branches | patience={BRANCH_PATIENCE} | {N_RESTARTS}×{MAX_FLIPS//N_RESTARTS//1000}K restarts')
print(f'  Top-{TOP_K_PARTICLES} particles × {len(MODES)} modes | GravityV3 momentum')
print('=' * 110)
print(f"  {'α':>5} | {'n':>5} | {'S1':>3} | {'Conf%':>6} | {'MFLS%':>6} | "
      f"{'QS%':>6} | {'Ens%':>6} | {'H11bE':>6} | {'H10b':>5} | {'ΔvsH11b':>7} | Time")
print('  ' + '-' * 100)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()
        s1_count = 0

        mode_ok   = {m: 0 for m in MODES}
        mode_tried = {m: 0 for m in MODES}
        mode_flips = {m: 0 for m in MODES}
        tot_viols = 0
        ens_ok = 0
        fc = 0

        for inst in range(N_INST):
            clauses = generate_3sat_instance(n_var, m_cls)
            eng = BSDTGravityV3(n_var, clauses)

            # ── Run GravityV3 (momentum + plateau escape) ──
            sf = eng.gravity_flow(steps=STEPS, particles=PARTICLES)

            # ── Get top-K particles ──
            top_indices, top_viols = eng.find_best_particles(sf, k=TOP_K_PARTICLES)

            any_solved = False

            for rank, (pi, viols) in enumerate(zip(top_indices, top_viols)):
                if viols == 0:
                    s1_count += 1
                    for md in MODES:
                        mode_ok[md] += 1
                    any_solved = True
                    break

                if rank == 0:
                    tot_viols += viols
                    fc += 1

                x_np = (sf[pi] > 0).cpu().numpy().astype(np.int32)
                w_conf = eng.get_confidence(sf, pi)
                w_mfls = eng.get_mfls(sf, pi)
                w_qs   = eng.get_quadsurf(sf, pi)
                weights = {'confidence': w_conf, 'mfls': w_mfls, 'quadsurf': w_qs}

                for md in MODES:
                    sol, flips = bpr_treewalk(
                        eng.clauses_v, eng.clauses_s,
                        x_np.copy(), weights[md],
                        max_flips=MAX_FLIPS, T_init=0.5, T_min=0.01,
                        p_random=0.1, beta=BETA, n_restarts=N_RESTARTS,
                        branch_patience=BRANCH_PATIENCE
                    )
                    sol_t = torch.tensor(sol, dtype=torch.long, device=device)
                    sat, _ = eng.check_single(sol_t)
                    if sat:
                        if rank == 0:
                            mode_ok[md] += 1
                            mode_flips[md] += flips
                            mode_tried[md] += 1
                        any_solved = True
                    elif rank == 0:
                        mode_flips[md] += flips
                        mode_tried[md] += 1

                if any_solved:
                    break

            if any_solved:
                ens_ok += 1

            if (inst + 1) % 10 == 0:
                el = time.time() - t0
                c_ = mode_ok['confidence'] / (inst+1) * 100
                m_ = mode_ok['mfls'] / (inst+1) * 100
                q_ = mode_ok['quadsurf'] / (inst+1) * 100
                e_ = ens_ok / (inst+1) * 100
                print(f'    α={alpha}, n={n_var}: {inst+1}/{N_INST}'
                      f'  C={c_:.0f}% M={m_:.0f}% Q={q_:.0f}% E={e_:.0f}%'
                      f'  ({el:.0f}s, ~{el/(inst+1)*N_INST:.0f}s total)')

        elapsed = time.time() - t0
        avg_v = tot_viols / max(fc, 1)

        for md in MODES:
            pct = mode_ok[md] / N_INST * 100
            af = mode_flips[md] // max(mode_tried[md], 1)
            results[md][(alpha, n_var)] = {'pct': pct, 'flips': af}

        ens_pct = ens_ok / N_INST * 100
        ensemble_results[(alpha, n_var)] = ens_pct
        s1_results[(alpha, n_var)] = s1_count

        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']

        h11e = h11b_ens[(alpha, n_var)]
        delta_ens = ens_pct - h11e
        ws_ref = h10b[(alpha, n_var)]

        tag = '★' if ens_pct >= 95 else ('▲' if delta_ens > 2 else
              ('≈' if abs(delta_ens) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {s1_count:3d}  | '
              f'{c_:5.1f}% | {m_:5.1f}% | {q_:5.1f}% | {ens_pct:5.1f}% | '
              f'{h11e:5.1f}% | {ws_ref:4.0f}% | {delta_ens:+6.1f}% | '
              f'{elapsed:4.0f}s {tag}')
    print('  ' + '-' * 100)


# ══════════════════════════════════════════════════════════════════════
#  Full Summary
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 110)
print('FULL COMPARISON — H9b → H10b → H11b → H12')
print('=' * 110)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H10b':>5} | {'H11bE':>6} | "
      f"{'H12C':>5} | {'H12M':>5} | {'H12Q':>5} | {'H12 Ens':>8} | {'Δ':>6}")
print('  ' + '-' * 85)
for alpha in ALPHAS:
    for n_var in NS:
        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']
        e_ = ensemble_results[(alpha, n_var)]
        h9  = h9b[(alpha, n_var)]
        h10 = h10b[(alpha, n_var)]
        h11e = h11b_ens[(alpha, n_var)]
        delta = e_ - h11e
        print(f'  {alpha:5.1f} | {n_var:5d} | {h9:4.0f}% | {h10:4.0f}% | '
              f'{h11e:5.1f}% | {c_:4.1f}% | {m_:4.1f}% | {q_:4.1f}% | '
              f'{e_:7.1f}% | {delta:+5.1f}%')
    print('  ' + '-' * 85)


# ══════════════════════════════════════════════════════════════════════
#  Charts
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating charts...\n')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
w = 0.15
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6']
labels_c = ['H9b (baseline)', 'H10b (WalkSAT)', 'H11b (BPR ens)',
            'H12 best single', 'H12 ensemble']

for i, n_var in enumerate(NS):
    ax = axes[i]
    x = np.arange(len(ALPHAS))

    bars_data = [
        [h9b[(a, n_var)] for a in ALPHAS],
        [h10b[(a, n_var)] for a in ALPHAS],
        [h11b_ens[(a, n_var)] for a in ALPHAS],
        [max(results['confidence'][(a, n_var)]['pct'],
             results['mfls'][(a, n_var)]['pct'],
             results['quadsurf'][(a, n_var)]['pct']) for a in ALPHAS],
        [ensemble_results[(a, n_var)] for a in ALPHAS],
    ]

    for k, (label, vals) in enumerate(zip(labels_c, bars_data)):
        offset = (k - 2) * w
        ax.bar(x + offset, vals, w, label=label, color=colors[k], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('H12: Tree-Walk BPR — Anti-Fragmentation SAT Solver\n'
             'Walk along branches of clause-variable tree | Switch when signal weakens',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('h12_treewalk_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h12_treewalk_results.png')

# ── Flip efficiency ──────────────────────────────────────────────────
print('\nFlip counts (lower = more efficient):')
print(f"  {'α':>5} | {'n':>5} | {'Conf':>8} | {'MFLS':>8} | {'QS':>8}")
print('  ' + '-' * 45)
for alpha in ALPHAS:
    for n_var in NS:
        cf = results['confidence'][(alpha, n_var)]['flips']
        mf = results['mfls'][(alpha, n_var)]['flips']
        qf = results['quadsurf'][(alpha, n_var)]['flips']
        print(f'  {alpha:5.1f} | {n_var:5d} | {cf:8d} | {mf:8d} | {qf:8d}')
    print('  ' + '-' * 45)

# ── Stage 1 improvement ─────────────────────────────────────────────
print('\nStage 1 (gravity-only): momentum + plateau vs no-momentum')
for alpha in ALPHAS:
    for n_var in NS:
        s1 = s1_results[(alpha, n_var)]
        if s1 > 0:
            print(f'  α={alpha}, n={n_var}: {s1}/50 solved by gravity alone ★')